## Assignment experiments

The following sections contain the decision-tree baseline, custom bagging/OOB analysis, SAMME implementation, robustness tests, bias–variance experiments, and their visualizations. The implementation loads `olivetti_faces_final.npz` created above.

SECTION 1.A CODE, 15 COMBINATIONS OF DECISION TREE

In [1]:
import numpy as np
import pandas as pd
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score
import time

In [2]:
data = np.load("olivetti_faces_final.npz" , allow_pickle = False)

X_train, y_train = data["X_train"], data["y_train"]
X_val, y_val = data["X_val"], data["y_val"]
X_test, y_test = data["X_test"], data["y_test"]

In [3]:
depth_values = [2,4,8,16,None]
leaf_values = [1,2,4]

In [4]:
results = []
models = {}

for depth in depth_values:
    for min_leaf in leaf_values:
        model = DecisionTreeClassifier(
            max_depth=depth,
            min_samples_leaf=min_leaf,
            random_state=42
        )
        model.fit(X_train, y_train)

        train_pred = model.predict(X_train)
        val_pred = model.predict(X_val)
        test_pred = model.predict(X_test)

        leaf_ids = model.apply(X_train)
        _, samples_per_leaf = np.unique(
            leaf_ids, return_counts=True
        )

        key = (depth, min_leaf)
        models[key] = model

        results.append({
            "max_depth": depth,
            "min_samples_leaf": min_leaf,
            "train_accuracy": accuracy_score(y_train, train_pred),
            "val_accuracy": accuracy_score(y_val, val_pred),
            "test_accuracy": accuracy_score(y_test, test_pred),
            "n_leaves": model.get_n_leaves(),
            "mean_samples_per_leaf": samples_per_leaf.mean(),
            "fraction_singleton_leaves": np.mean(
                samples_per_leaf == 1
            )
        })

In [5]:
results_df = pd.DataFrame(results)


# Select using validation accuracy only
best_index = results_df["val_accuracy"].idxmax()
best_row = results_df.loc[best_index]

# Recover the original Python None value from the grid
best_depth = depth_values[
    best_index // len(leaf_values)
]
best_leaf = int(best_row["min_samples_leaf"])

best_tree = models[(best_depth, best_leaf)]

print("Frozen tree settings")
print("max_depth:", best_depth)
print("min_samples_leaf:", best_leaf)
print("Train accuracy:", round(best_row["train_accuracy"], 4))
print("Validation accuracy:", round(best_row["val_accuracy"], 4))
print("Test accuracy:", round(best_row["test_accuracy"], 4))
print("Number of leaves:", int(best_row["n_leaves"]))
print("Mean samples per leaf:",
      round(best_row["mean_samples_per_leaf"], 3))
print("Fraction of singleton leaves:",
      round(best_row["fraction_singleton_leaves"], 4))


Frozen tree settings
max_depth: None
min_samples_leaf: 2
Train accuracy: 0.9792
Validation accuracy: 0.6125
Test accuracy: 0.6
Number of leaves: 46
Mean samples per leaf: 5.217
Fraction of singleton leaves: 0.0


In [6]:
print(results_df[[
    "max_depth",
    "min_samples_leaf",
    "val_accuracy"
]])

print("\nMissing values:")
print(results_df.isna().sum())

    max_depth  min_samples_leaf  val_accuracy
0         2.0                 1        0.0750
1         2.0                 2        0.0750
2         2.0                 4        0.0750
3         4.0                 1        0.1250
4         4.0                 2        0.1250
5         4.0                 4        0.1250
6         8.0                 1        0.2000
7         8.0                 2        0.2000
8         8.0                 4        0.2000
9        16.0                 1        0.3625
10       16.0                 2        0.3625
11       16.0                 4        0.3625
12        NaN                 1        0.5875
13        NaN                 2        0.6125
14        NaN                 4        0.6000

Missing values:
max_depth                    3
min_samples_leaf             0
train_accuracy               0
val_accuracy                 0
test_accuracy                0
n_leaves                     0
mean_samples_per_leaf        0
fraction_singleton_leaves    0

SECTION 1.B BAGGING FROM SCRATCH

In [7]:
best_depth = None
best_leaf = 4

B_values = [1,5,10,25,50,100,200]
rng = np.random.default_rng(2026)
n_train = len(X_train)



In [ ]:
trees = []
bootstrap_indices = []
distinct_counts = []

for b in range(max(B_values)):
    sample_idx = rng.choice( n_train, size = n_train, replace =True)
    tree = DecisionTreeClassifier( max_depth = best_depth, min_samples_leaf = best_leaf, random_state = 1000+b)
    tree.fit(X_train[sample_idx], y_train[sample_idx])
    trees.append(tree)
    bootstrap_indices.append(sample_idx)
    distinct_counts.append(len(np.unique(sample_idx)))

print("Trees trained:", len(trees))
print("First bootstrap sample size:", len(bootstrap_indices[0]))
print("Distinct examples in first sample:", distinct_counts[0])

In [ ]:
#majority voting

train_tree_preds = np.array([ tree.predict(X_train) for tree in trees ])
val_tree_preds = np.array([tree.predict(X_val) for tree in trees])
test_tree_preds = np.array([tree.predict(X_test) for tree in trees])
print("Train predictions:", train_tree_preds.shape)
print("Validation predictions:", val_tree_preds.shape)
print("Test predictions:", test_tree_preds.shape)

In [ ]:
def majority_vote(pred):
    classes = np.unique(y_train)
    n_images = pred.shape[1]
    final_predictions = np.empty(n_images, dtype=y_train.dtype)

    for j in range(n_images):
        votes = [ np.sum(pred[:, j] == cls) for cls in classes]
        final_predictions[j] = classes[np.argmax(votes)]
        return final_predictions

example_preds = majority_vote(val_tree_preds[:5])

print("First 10 validation predictions:", example_preds[:10])
print("First 10 true labels:", y_val[:10])

In [ ]:

# Theoretical expected number of distinct examples in one bootstrap sample
expected_distinct = n_train * (
    1 - (1 - 1 / n_train) ** n_train
)

bagging_results = []

for B in B_values:

    # Combine predictions from the first B trees
    train_pred = majority_vote(train_tree_preds[:B])
    val_pred = majority_vote(val_tree_preds[:B])
    test_pred = majority_vote(test_tree_preds[:B])

    # Average distinct training examples across these B samples
    avg_distinct = np.mean(distinct_counts[:B])

    bagging_results.append({
        "B": B,
        "train_accuracy": accuracy_score(y_train, train_pred),
        "val_accuracy": accuracy_score(y_val, val_pred),
        "test_accuracy": accuracy_score(y_test, test_pred),
        "avg_distinct_examples": avg_distinct,
        "theoretical_distinct_examples": expected_distinct
    })

bagging_df = pd.DataFrame(bagging_results)

display(bagging_df.round(4))


In [ ]:

print(f"Theoretical expected distinct examples: {expected_distinct:.3f}")
print(f"Training set size: {n_train}")

print("\nBagging results:")
display(
    bagging_df.style.format({
        "train_accuracy": "{:.4f}",
        "val_accuracy": "{:.4f}",
        "test_accuracy": "{:.4f}",
        "avg_distinct_examples": "{:.2f}",
        "theoretical_distinct_examples": "{:.2f}"
    })
)


In [ ]:

# Check that labels and saved split arrays are consistent
print("Unique training labels:", np.unique(y_train))
print("Unique validation labels:", np.unique(y_val))
print("Unique test labels:", np.unique(y_test))

print("\nPrediction array shapes:")
print("Train:", train_tree_preds.shape)
print("Validation:", val_tree_preds.shape)
print("Test:", test_tree_preds.shape)

# Check prediction distributions for a few ensemble sizes
for B in [1, 5, 10, 25, 50, 100, 200]:
    val_pred = majority_vote(val_tree_preds[:B])
    test_pred = majority_vote(test_tree_preds[:B])

    print(f"\nB = {B}")
    print("Validation predicted identities:",
          np.unique(val_pred, return_counts=True))
    print("Test predicted identities:",
          np.unique(test_pred, return_counts=True))


1.C OUT OF BAG ACCURACIES AND PLOT

In [ ]:
# Check that Q1(b) variables exist
required_variables = [
    "trees",
    "bootstrap_indices",
    "train_tree_preds",
    "val_tree_preds",
    "test_tree_preds",
    "B_values",
    "X_train",
    "y_train",
    "X_val",
    "y_val",
    "X_test",
    "y_test",
    "majority_vote"
]

missing = [name for name in required_variables if name not in globals()]

if missing:
    print("Missing variables:", missing)
    print("Run the relevant Q1(b) cells before continuing.")
else:
    print("All required Q1(b) variables are available.")
    print("Number of trees:", len(trees))
    print("Bootstrap samples:", len(bootstrap_indices))
    print("Training prediction shape:", train_tree_preds.shape)
    print("Validation prediction shape:", val_tree_preds.shape)
    print("Test prediction shape:", test_tree_preds.shape)

In [ ]:
import numpy as np
import pandas as pd

oob_predictions_by_B = {}
oob_accuracies = []

for B in B_values:
    oob_pred = np.full(len(y_train), -1, dtype=int)

    for i in range(len(y_train)):
        # Trees among the first B whose bootstrap sample excludes image i
        eligible_trees = [
            j for j in range(B)
            if i not in bootstrap_indices[j]
        ]

        if eligible_trees:
            votes = [
                trees[j].predict(X_train[i:i+1])[0]
                for j in eligible_trees
            ]

            classes, counts = np.unique(votes, return_counts=True)
            oob_pred[i] = classes[np.argmax(counts)]

    # Only score examples with at least one OOB prediction
    valid = oob_pred != -1

    oob_accuracy = np.mean(oob_pred[valid] == y_train[valid])

    oob_predictions_by_B[B] = oob_pred
    oob_accuracies.append(oob_accuracy)

    print(
        f"B = {B:3d} | "
        f"OOB-covered images = {valid.sum():3d}/{len(y_train)} | "
        f"OOB accuracy = {oob_accuracy:.4f}"
    )

In [ ]:
from itertools import combinations

pairwise_disagreements = []
mean_vote_margins = []

def calculate_disagreement(predictions):
    # predictions shape: (number of trees, number of images)
    n_trees = predictions.shape[0]

    if n_trees < 2:
        return 0.0

    disagreements = []

    for i, j in combinations(range(n_trees), 2):
        disagreements.append(np.mean(predictions[i] != predictions[j]))

    return np.mean(disagreements)


def calculate_mean_margin(predictions, classes):
    margins = []

    for image_index in range(predictions.shape[1]):
        votes = np.array([
            np.sum(predictions[:, image_index] == cls)
            for cls in classes
        ])

        # Two largest vote counts: winner and runner-up
        top_two = np.sort(votes)[-2:]
        margin = (top_two[-1] - top_two[-2]) / predictions.shape[0]
        margins.append(margin)

    return np.mean(margins)


classes = np.unique(y_train)

for B in B_values:
    val_predictions = val_tree_preds[:B]

    disagreement = calculate_disagreement(val_predictions)
    margin = calculate_mean_margin(val_predictions, classes)

    pairwise_disagreements.append(disagreement)
    mean_vote_margins.append(margin)

    print(
        f"B = {B:3d} | "
        f"Disagreement = {disagreement:.4f} | "
        f"Mean vote margin = {margin:.4f}"
    )

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score

q1c_results = pd.DataFrame({
    "B": B_values,
    "OOB_accuracy": oob_accuracies,
    "Validation_accuracy": [
        accuracy_score(y_val, majority_vote(val_tree_preds[:B]))
        for B in B_values
    ],
    "Pairwise_disagreement": pairwise_disagreements,
    "Mean_vote_margin": mean_vote_margins
})

display(q1c_results.round(4))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].plot(
    q1c_results["B"],
    q1c_results["OOB_accuracy"],
    marker="o"
)
axes[0].set_title("OOB Accuracy")
axes[0].set_xlabel("Number of trees (B)")
axes[0].set_ylabel("Accuracy")
axes[0].set_ylim(0, 1)
axes[0].grid(True)

axes[1].plot(
    q1c_results["B"],
    q1c_results["Pairwise_disagreement"],
    marker="o"
)
axes[1].set_title("Pairwise Disagreement")
axes[1].set_xlabel("Number of trees (B)")
axes[1].set_ylabel("Disagreement")
axes[1].set_ylim(0, 1)
axes[1].grid(True)

axes[2].plot(
    q1c_results["B"],
    q1c_results["Mean_vote_margin"],
    marker="o"
)
axes[2].set_title("Mean Vote Margin")
axes[2].set_xlabel("Number of trees (B)")
axes[2].set_ylabel("Mean margin")
axes[2].set_ylim(0, 1)
axes[2].grid(True)

plt.tight_layout()
plt.show()

In [ ]:
best_row_index = q1c_results["Validation_accuracy"].idxmax()
best_B = int(q1c_results.loc[best_row_index, "B"])

print("Selected ensemble size:", best_B)
print(
    "Validation accuracy:",
    round(q1c_results.loc[best_row_index, "Validation_accuracy"], 4)
)
print(
    "OOB accuracy:",
    round(q1c_results.loc[best_row_index, "OOB_accuracy"], 4)
)
print(
    "Pairwise disagreement:",
    round(q1c_results.loc[best_row_index, "Pairwise_disagreement"], 4)
)
print(
    "Mean vote margin:",
    round(q1c_results.loc[best_row_index, "Mean_vote_margin"], 4)
)

1.D testing on masked sets and comparision of decision tree and bagging model

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.metrics import accuracy_score

# Load the saved dataset and split
data = np.load("olivetti_faces_final.npz")

X_test = data["X_test"]
y_test = data["y_test"]
X_test_occluded = data["X_test_occluded"]
y_test_occluded = data["y_test_occluded"]

# Confirm required models and settings exist
required = ["best_tree", "trees", "best_B"]
missing = [name for name in required if name not in globals()]

if missing:
    raise NameError(
        f"Missing variables: {missing}. "
        "Run the relevant Q1(a), Q1(b), and Q1(c) cells first."
    )

print("Clean test shape:", X_test.shape)
print("Occluded test shape:", X_test_occluded.shape)
print("Selected ensemble size:", best_B)

In [ ]:
# Single decision tree predictions
tree_clean_pred = best_tree.predict(X_test)
tree_occluded_pred = best_tree.predict(X_test_occluded)

# Predictions from the first best_B trees in the bagging ensemble
def get_vote_counts(predictions, classes):
    return np.array([
        [np.sum(predictions[:, i] == cls) for cls in classes]
        for i in range(predictions.shape[1])
    ])

classes = np.unique(y_test)

# Obtain each tree's predictions on both test sets
clean_tree_predictions = np.array([
    trees[j].predict(X_test) for j in range(best_B)
])

occluded_tree_predictions = np.array([
    trees[j].predict(X_test_occluded) for j in range(best_B)
])

clean_votes = get_vote_counts(clean_tree_predictions, classes)
occluded_votes = get_vote_counts(occluded_tree_predictions, classes)

# Convert vote counts to the ensemble's predicted identity
bag_clean_pred = classes[np.argmax(clean_votes, axis=1)]
bag_occluded_pred = classes[np.argmax(occluded_votes, axis=1)]

results_1d = pd.DataFrame({
    "Model": ["Single tree", "Bagging"],
    "Clean test accuracy": [
        accuracy_score(y_test, tree_clean_pred),
        accuracy_score(y_test, bag_clean_pred)
    ],
    "Occluded test accuracy": [
        accuracy_score(y_test_occluded, tree_occluded_pred),
        accuracy_score(y_test_occluded, bag_occluded_pred)
    ]
})

display(results_1d.round(4))

In [ ]:
def tree_confidence(model, X):
    probabilities = model.predict_proba(X)
    return np.max(probabilities, axis=1)


def bagging_confidence(vote_counts):
    # Fraction of trees that voted for the winning identity
    return np.max(vote_counts, axis=1) / vote_counts.sum(axis=1)


confidence_results = pd.DataFrame({
    "Model": ["Single tree", "Bagging"],
    "Clean mean confidence": [
        tree_confidence(best_tree, X_test).mean(),
        bagging_confidence(clean_votes).mean()
    ],
    "Occluded mean confidence": [
        tree_confidence(best_tree, X_test_occluded).mean(),
        bagging_confidence(occluded_votes).mean()
    ]
})

display(confidence_results.round(4))

In [ ]:
corrected_indices = np.where(
    (tree_clean_pred != y_test) &
    (bag_clean_pred == y_test)
)[0]

print("Number of clean test images corrected by bagging:",
      len(corrected_indices))

examples_corrected = corrected_indices[:2]

for i in examples_corrected:
    print(
        f"Image index: {i} | True identity: {y_test[i]} | "
        f"Tree prediction: {tree_clean_pred[i]} | "
        f"Bagging prediction: {bag_clean_pred[i]}"
    )

In [ ]:
if len(examples_corrected) > 0:
    fig, axes = plt.subplots(
        1, len(examples_corrected),
        figsize=(5 * len(examples_corrected), 4)
    )

    if len(examples_corrected) == 1:
        axes = [axes]

    for ax, i in zip(axes, examples_corrected):
        ax.imshow(X_test[i].reshape(64, 64), cmap="gray")
        ax.set_title(
            f"True: {y_test[i]}\n"
            f"Tree: {tree_clean_pred[i]}\n"
            f"Bagging: {bag_clean_pred[i]}"
        )
        ax.axis("off")

    plt.tight_layout()
    plt.show()
else:
    print("No corrected examples were found on the clean test set.")

In [ ]:
both_wrong_indices = np.where(
    (tree_clean_pred != y_test) &
    (bag_clean_pred != y_test)
)[0]

print("Number of clean test images both models get wrong:",
      len(both_wrong_indices))

examples_both_wrong = both_wrong_indices[:2]

for i in examples_both_wrong:
    print(
        f"Image index: {i} | True identity: {y_test[i]} | "
        f"Tree prediction: {tree_clean_pred[i]} | "
        f"Bagging prediction: {bag_clean_pred[i]}"
    )

In [ ]:
if len(examples_both_wrong) > 0:
    fig, axes = plt.subplots(
        1, len(examples_both_wrong),
        figsize=(5 * len(examples_both_wrong), 4)
    )

    if len(examples_both_wrong) == 1:
        axes = [axes]

    for ax, i in zip(axes, examples_both_wrong):
        ax.imshow(X_test[i].reshape(64, 64), cmap="gray")
        ax.set_title(
            f"True: {y_test[i]}\n"
            f"Tree: {tree_clean_pred[i]}\n"
            f"Bagging: {bag_clean_pred[i]}"
        )
        ax.axis("off")

    plt.tight_layout()
    plt.show()
else:
    print("No examples where both models are wrong were found.")

In [ ]:
selected_examples = list(examples_corrected) + list(examples_both_wrong)

for i in selected_examples:
    print(f"\nTest image index: {i}")
    print("True identity:", y_test[i])
    print("Single tree prediction:", tree_clean_pred[i])
    print("Bagging prediction:", bag_clean_pred[i])
    print("Bagging vote counts by identity:")

    vote_table = pd.DataFrame({
        "Identity": classes,
        "Votes": clean_votes[i]
    })

    vote_table = vote_table[vote_table["Votes"] > 0]
    vote_table = vote_table.sort_values(
        "Votes", ascending=False
    )

    display(vote_table.head(10).reset_index(drop=True))

In [ ]:
tree_clean_acc = accuracy_score(y_test, tree_clean_pred)
tree_occluded_acc = accuracy_score(y_test_occluded, tree_occluded_pred)

bag_clean_acc = accuracy_score(y_test, bag_clean_pred)
bag_occluded_acc = accuracy_score(y_test_occluded, bag_occluded_pred)

print(f"Single tree accuracy drop: "
      f"{tree_clean_acc - tree_occluded_acc:.4f}")

print(f"Bagging accuracy drop: "
      f"{bag_clean_acc - bag_occluded_acc:.4f}")

QUESTION 2A SAMME BOOSTING ON SAMPLE DATASET

In [ ]:
import numpy as np
from sklearn.tree import DecisionTreeClassifier

# Nine samples, three classes: 0, 1, 2
X_toy = np.array([
    [0.0, 0.0],
    [0.1, 0.0],
    [0.2, 0.1],
    [1.0, 0.0],
    [1.1, 0.1],
    [1.2, 0.0],
    [0.0, 1.0],
    [0.1, 1.1],
    [0.2, 1.2]
])

y_toy = np.array([0, 0, 0, 1, 1, 1, 2, 2, 2])

K_toy = len(np.unique(y_toy))
sample_weights = np.ones(len(y_toy)) / len(y_toy)

print("Number of samples:", len(y_toy))
print("Number of classes:", K_toy)
print("Initial sample weights:", sample_weights)
print("Initial weight sum:", sample_weights.sum())

In [ ]:
toy_tree = DecisionTreeClassifier(max_depth=1, random_state=7)
toy_tree.fit(X_toy, y_toy, sample_weight=sample_weights)

toy_predictions = toy_tree.predict(X_toy)

# Weighted classification error
incorrect = toy_predictions != y_toy
weighted_error = np.sum(sample_weights * incorrect)

# SAMME weak learner threshold
random_error_threshold = 1 - 1 / K_toy

print("Initial sample weights:")
print(np.round(sample_weights, 4))

print("\nTrue labels:")
print(y_toy)

print("\nWeak learner predictions:")
print(toy_predictions)

print("\nIncorrect predictions:")
print(incorrect)

print("\nWeighted error:", round(weighted_error, 6))
print("Random-guess threshold:", round(random_error_threshold, 6))

if weighted_error >= random_error_threshold:
    print(
        "\nThis weak learner is no better than random multiclass prediction. "
        "SAMME stops without accepting this learner."
    )
else:
    alpha_t = (
        np.log((1 - weighted_error) / weighted_error)
        + np.log(K_toy - 1)
    )

    # SAMME update: increase weights for misclassified samples
    updated_weights = sample_weights * np.exp(
        alpha_t * incorrect.astype(float)
    )

    # Normalize so weights sum to one
    updated_weights /= updated_weights.sum()

    print("\nLearner weight alpha_t:", round(alpha_t, 6))
    print("\nUpdated normalized sample weights:")
    print(np.round(updated_weights, 6))
    print("Updated weight sum:", updated_weights.sum())

SAMME IMPLEMENTATION

In [ ]:
import time
import numpy as np
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score


class SAMMEBoost:
    def __init__(self, max_depth=1, n_estimators=50, random_state=42):
        self.max_depth = max_depth
        self.n_estimators = n_estimators
        self.random_state = random_state

        self.estimators_ = []
        self.estimator_weights_ = []
        self.classes_ = None
        self.history_ = []
        self.sample_weights_ = None
        self.stop_reason_ = None

    def fit(self, X, y):
        self.classes_ = np.unique(y)
        K = len(self.classes_)
        n = len(y)

        # Uniform initial weights
        weights = np.ones(n, dtype=float) / n

        self.estimators_ = []
        self.estimator_weights_ = []
        self.history_ = []
        self.stop_reason_ = None

        rng = np.random.default_rng(self.random_state)

        for round_index in range(self.n_estimators):
            tree = DecisionTreeClassifier(
                max_depth=self.max_depth,
                random_state=int(rng.integers(0, 2**31 - 1))
            )

            tree.fit(X, y, sample_weight=weights)
            predictions = tree.predict(X)

            incorrect = predictions != y
            weighted_error = float(np.sum(weights * incorrect))

            # Reject learners that are no better than random
            if weighted_error >= 1 - 1 / K:
                self.stop_reason_ = (
                    f"Round {round_index + 1}: weighted error "
                    f"{weighted_error:.6f} >= random threshold "
                    f"{1 - 1/K:.6f}"
                )
                break

            # Perfect classification: accept and stop after recording
            if weighted_error <= 1e-15:
                alpha = 1.0  # finite positive weight for a perfect learner
                self.estimators_.append(tree)
                self.estimator_weights_.append(alpha)

                self.history_.append({
                    "round": round_index + 1,
                    "weighted_error": weighted_error,
                    "alpha": alpha,
                    "max_sample_weight": float(weights.max()),
                    "top10_weight": float(np.sort(weights)[-min(10, n):].sum()),
                    "sample_weights": weights.copy()
                })

                self.stop_reason_ = "Perfect weak learner; stopped early."
                break

            alpha = (
                np.log((1 - weighted_error) / weighted_error)
                + np.log(K - 1)
            )

            # Store this accepted learner
            self.estimators_.append(tree)
            self.estimator_weights_.append(float(alpha))

            # SAMME weight update
            weights *= np.exp(alpha * incorrect.astype(float))
            weights /= weights.sum()

            self.history_.append({
                "round": round_index + 1,
                "weighted_error": weighted_error,
                "alpha": float(alpha),
                "max_sample_weight": float(weights.max()),
                "top10_weight": float(np.sort(weights)[-min(10, n):].sum()),
                "sample_weights": weights.copy()
            })

        self.sample_weights_ = weights.copy()

        if self.stop_reason_ is None:
            self.stop_reason_ = "Requested number of rounds completed."

        return self

    def decision_function(self, X):
        # Scores are accumulated weighted votes, one column per class
        scores = np.zeros((len(X), len(self.classes_)), dtype=float)

        class_to_index = {
            cls: index for index, cls in enumerate(self.classes_)
        }

        for tree, alpha in zip(
            self.estimators_, self.estimator_weights_
        ):
            predictions = tree.predict(X)

            for cls, index in class_to_index.items():
                scores[:, index] += alpha * (predictions == cls)

        return scores

    def predict_proba(self, X):
        scores = self.decision_function(X)
        total_alpha = np.sum(self.estimator_weights_)

        if total_alpha <= 0:
            return np.ones_like(scores) / scores.shape[1]

        # Assignment-defined normalized weighted vote
        return scores / total_alpha

    def predict(self, X):
        scores = self.decision_function(X)
        return self.classes_[np.argmax(scores, axis=1)]

In [ ]:
depth_values = [1, 2, 3, 4]
round_values = [10, 25, 50, 100]

boost_models = {}
boost_results = []

for depth in depth_values:
    for n_rounds in round_values:
        model = SAMMEBoost(
            max_depth=depth,
            n_estimators=n_rounds,
            random_state=2026
        )

        start = time.perf_counter()
        model.fit(X_train, y_train)
        elapsed = time.perf_counter() - start

        train_pred = model.predict(X_train)
        val_pred = model.predict(X_val)
        test_pred = model.predict(X_test)

        boost_models[(depth, n_rounds)] = model

        boost_results.append({
            "depth": depth,
            "requested_rounds": n_rounds,
            "actual_rounds": len(model.estimators_),
            "train_accuracy": accuracy_score(y_train, train_pred),
            "val_accuracy": accuracy_score(y_val, val_pred),
            "test_accuracy": accuracy_score(y_test, test_pred),
            "training_time_seconds": elapsed,
            "stop_reason": model.stop_reason_
        })

boost_df = pd.DataFrame(boost_results)
display(boost_df.round(4))

In [ ]:
best_index = boost_df["val_accuracy"].idxmax()
best_boost_row = boost_df.loc[best_index]

best_depth = int(best_boost_row["depth"])
best_rounds = int(best_boost_row["requested_rounds"])
best_boost = boost_models[(best_depth, best_rounds)]

print("Selected tree depth:", best_depth)
print("Selected requested rounds:", best_rounds)
print("Actual accepted rounds:", len(best_boost.estimators_))
print("Validation accuracy:", round(best_boost_row["val_accuracy"], 4))
print("Training accuracy:", round(best_boost_row["train_accuracy"], 4))
print("Clean test accuracy:", round(best_boost_row["test_accuracy"], 4))
print("Stop reason:", best_boost.stop_reason_)

In [ ]:
history_df = pd.DataFrame([
    {
        "round": h["round"],
        "weighted_training_error": h["weighted_error"],
        "learner_weight": h["alpha"],
        "max_sample_weight": h["max_sample_weight"],
        "top10_weight": h["top10_weight"]
    }
    for h in best_boost.history_
])

display(history_df.head(10).round(5))

print("Maximum sample weight across rounds:",
      history_df["max_sample_weight"].max())
print("Maximum top-10 total weight:",
      history_df["top10_weight"].max())

assert history_df["max_sample_weight"].between(0, 1).all()
assert history_df["top10_weight"].between(0, 1).all()

fig, axes = plt.subplots(2, 2, figsize=(12, 8))

axes[0, 0].plot(
    history_df["round"], history_df["weighted_training_error"],
    marker="o"
)
axes[0, 0].set_title("Weighted Training Error")
axes[0, 0].set_xlabel("Round")
axes[0, 0].set_ylabel("Weighted error")
axes[0, 0].grid(True)

axes[0, 1].plot(
    history_df["round"], history_df["learner_weight"],
    marker="o"
)
axes[0, 1].set_title("Weak Learner Weight")
axes[0, 1].set_xlabel("Round")
axes[0, 1].set_ylabel("Alpha")
axes[0, 1].grid(True)

axes[1, 0].plot(
    history_df["round"], history_df["max_sample_weight"],
    marker="o"
)
axes[1, 0].set_title("Maximum Sample Weight")
axes[1, 0].set_xlabel("Round")
axes[1, 0].set_ylabel("Weight")
axes[1, 0].grid(True)

axes[1, 1].plot(
    history_df["round"], history_df["top10_weight"],
    marker="o"
)
axes[1, 1].set_title("Weight on 10 Heaviest Images")
axes[1, 1].set_xlabel("Round")
axes[1, 1].set_ylabel("Total weight")
axes[1, 1].grid(True)

plt.tight_layout()
plt.show()

2.C BAGGING VS BOOSTING

In [ ]:
import time
import numpy as np
import pandas as pd
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

comparison_sizes = [25, 50, 100]

# Use the frozen Q1(a) tree settings
bag_depth = best_depth if "best_depth" in globals() else None
bag_leaf = best_leaf if "best_leaf" in globals() else 4

# Use a separate, reproducible random generator for this comparison
bag_rng = np.random.default_rng(2026)
bag_comparison_models = {}
comparison_rows = []

n_train = len(y_train)

for B in comparison_sizes:
    bag_trees = []
    start = time.perf_counter()

    for b in range(B):
        sample_idx = bag_rng.choice(
            n_train, size=n_train, replace=True
        )

        tree = DecisionTreeClassifier(
            max_depth=bag_depth,
            min_samples_leaf=bag_leaf,
            random_state=1000 + b
        )
        tree.fit(X_train[sample_idx], y_train[sample_idx])
        bag_trees.append(tree)

    bag_time = time.perf_counter() - start
    bag_comparison_models[B] = bag_trees

    def bag_predict(model_list, X):
        classes = np.unique(y_train)
        predictions = np.array([tree.predict(X) for tree in model_list])
        votes = np.stack([
            np.sum(predictions == cls, axis=0) for cls in classes
        ])
        return classes[np.argmax(votes, axis=0)]

    bag_train_pred = bag_predict(bag_trees, X_train)
    bag_val_pred = bag_predict(bag_trees, X_val)
    bag_test_pred = bag_predict(bag_trees, X_test)

    comparison_rows.append({
        "Model": "Bagging",
        "Learners": B,
        "Train accuracy": accuracy_score(y_train, bag_train_pred),
        "Validation accuracy": accuracy_score(y_val, bag_val_pred),
        "Clean test accuracy": accuracy_score(y_test, bag_test_pred),
        "Training time (seconds)": bag_time
    })

# Compare against the boosting models trained in Q2(b)
for B in comparison_sizes:
    boost_model = boost_models[(best_depth, B)]

    comparison_rows.append({
        "Model": "SAMME boosting",
        "Learners": B,
        "Train accuracy": accuracy_score(y_train, boost_model.predict(X_train)),
        "Validation accuracy": accuracy_score(y_val, boost_model.predict(X_val)),
        "Clean test accuracy": accuracy_score(y_test, boost_model.predict(X_test)),
        "Training time (seconds)": boost_df.loc[
            (boost_df["depth"] == best_depth) &
            (boost_df["requested_rounds"] == B),
            "training_time_seconds"
        ].iloc[0]
    })

comparison_df = pd.DataFrame(comparison_rows)
display(comparison_df.round(4))

In [ ]:
final_weights = best_boost.sample_weights_
hard_local_idx = int(np.argmax(final_weights))

hard_face = X_train[hard_local_idx]
hard_label = y_train[hard_local_idx]

print("Training-set position:", hard_local_idx)
print("True identity:", hard_label)
print("Final sample weight:", final_weights[hard_local_idx])

plt.figure(figsize=(4, 4))
plt.imshow(hard_face.reshape(64, 64), cmap="gray")
plt.title(
    f"High-weight training face\n"
    f"True identity: {hard_label}\n"
    f"Weight: {final_weights[hard_local_idx]:.4f}"
)
plt.axis("off")
plt.show()

In [ ]:
prediction_history = []

for round_number, tree in enumerate(best_boost.estimators_, start=1):
    prediction = tree.predict(X_train[hard_local_idx:hard_local_idx + 1])[0]

    prediction_history.append({
        "round": round_number,
        "prediction": prediction,
        "correct": prediction == hard_label,
        "sample_weight_after_round": (
            best_boost.history_[round_number - 1]["sample_weights"][hard_local_idx]
            if "sample_weights" in best_boost.history_[round_number - 1]
            else np.nan
        )
    })

prediction_history_df = pd.DataFrame(prediction_history)
display(prediction_history_df.round(5))

2.D model on masked TEST SET

In [ ]:
# Frozen baseline from Q1(a)
tree_model = best_tree

# Frozen final bagging model from the comparison
final_bagging = bag_comparison_models[100]

# Frozen selected boosting model from Q2(b)
final_boosting = best_boost

def bag_scores(model_list, X, classes):
    individual_predictions = np.array([
        tree.predict(X) for tree in model_list
    ])
    votes = np.stack([
        np.sum(individual_predictions == cls, axis=0)
        for cls in classes
    ], axis=1)
    predicted = classes[np.argmax(votes, axis=1)]
    confidence = np.max(votes, axis=1) / len(model_list)
    return predicted, confidence, votes

def evaluate_model(model_name, model, X, y, classes):
    if model_name == "Single tree":
        pred = model.predict(X)
        confidence = np.max(model.predict_proba(X), axis=1)
    elif model_name == "Bagging":
        pred, confidence, _ = bag_scores(model, X, classes)
    else:
        pred = model.predict(X)
        confidence = np.max(model.predict_proba(X), axis=1)

    return {
        "Model": model_name,
        "Accuracy": accuracy_score(y, pred),
        "Mean winning vote/probability": np.mean(confidence)
    }

classes = np.unique(y_train)
evaluation_rows = []

for label, X_eval, y_eval in [
    ("Clean", X_test, y_test),
    ("Masked", X_test_occluded, y_test_occluded)
]:
    evaluation_rows.append({
        "Test set": label,
        **evaluate_model("Single tree", tree_model, X_eval, y_eval, classes)
    })
    evaluation_rows.append({
        "Test set": label,
        **evaluate_model("Bagging", final_bagging, X_eval, y_eval, classes)
    })
    evaluation_rows.append({
        "Test set": label,
        **evaluate_model("Boosting", final_boosting, X_eval, y_eval, classes)
    })

q2d_results = pd.DataFrame(evaluation_rows)
display(q2d_results.round(4))

In [ ]:
bag_clean_pred, bag_clean_conf, _ = bag_scores(
    final_bagging, X_test, classes
)
bag_mask_pred, bag_mask_conf, _ = bag_scores(
    final_bagging, X_test_occluded, classes
)

boost_clean_pred = final_boosting.predict(X_test)
boost_mask_pred = final_boosting.predict(X_test_occluded)

# Find examples where the two models differ in whether their prediction changes
different_response = np.where(
    (bag_clean_pred == bag_mask_pred) !=
    (boost_clean_pred == boost_mask_pred)
)[0]

print("Images with different responses to masking:", len(different_response))

if len(different_response) > 0:
    i = int(different_response[0])

    print("Test position:", i)
    print("True identity:", y_test[i])
    print("Bagging clean/masked:", bag_clean_pred[i], bag_mask_pred[i])
    print("Boosting clean/masked:", boost_clean_pred[i], boost_mask_pred[i])

    fig, axes = plt.subplots(1, 2, figsize=(7, 4))
    axes[0].imshow(X_test[i].reshape(64, 64), cmap="gray")
    axes[0].set_title("Clean")
    axes[1].imshow(X_test_occluded[i].reshape(64, 64), cmap="gray")
    axes[1].set_title("Masked")
    for ax in axes:
        ax.axis("off")
    plt.tight_layout()
    plt.show()
else:
    print("No image found where the models differ in prediction-change behavior.")

In [ ]:
# Inspect test position 2: clean versus masked
i = 2
classes = np.unique(y_train)

def show_bagging_votes(model_list, X, image_index, classes):
    predictions = np.array([
        tree.predict(X[image_index:image_index + 1])[0]
        for tree in model_list
    ])
    vote_counts = {
        int(cls): int(np.sum(predictions == cls))
        for cls in classes
        if np.any(predictions == cls)
    }
    return predictions, vote_counts

bag_clean_trees, bag_clean_vote_counts = show_bagging_votes(
    final_bagging, X_test, i, classes
)
bag_mask_trees, bag_mask_vote_counts = show_bagging_votes(
    final_bagging, X_test_occluded, i, classes
)

boost_clean_trees = np.array([
    tree.predict(X_test[i:i + 1])[0]
    for tree in final_boosting.estimators_
])
boost_mask_trees = np.array([
    tree.predict(X_test_occluded[i:i + 1])[0]
    for tree in final_boosting.estimators_
])

print("True identity:", y_test[i])

print("\nBagging clean votes:", bag_clean_vote_counts)
print("Bagging masked votes:", bag_mask_vote_counts)

print("\nBoosting clean weighted votes:")
print({
    int(cls): round(float(sum(
        alpha for pred, alpha in zip(
            boost_clean_trees, final_boosting.estimator_weights_
        ) if pred == cls
    )), 4)
    for cls in np.unique(boost_clean_trees)
})

print("\nBoosting masked weighted votes:")
print({
    int(cls): round(float(sum(
        alpha for pred, alpha in zip(
            boost_mask_trees, final_boosting.estimator_weights_
        ) if pred == cls
    )), 4)
    for cls in np.unique(boost_mask_trees)
})

print("\nFraction of bagging trees that change prediction:",
      np.mean(bag_clean_trees != bag_mask_trees))
print("Fraction of boosting trees that change prediction:",
      np.mean(boost_clean_trees != boost_mask_trees))

Question 3.A variance and bias

In [ ]:
import numpy as np
import pandas as pd
import time
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

data = np.load("olivetti_faces_final.npz")

X_train = data["X_train"]
y_train = data["y_train"]
X_test = data["X_test"]
y_test = data["y_test"]

X_test_occluded = data["X_test_occluded"]
y_test_occluded = data["y_test_occluded"]

classes = np.unique(y_train)
K = len(classes)
N = len(y_test)

# Selected models from earlier questions
selected_tree = best_tree
selected_boost = best_boost

# Q1(a) selected settings
selected_leaf = best_leaf

print("Training images:", len(y_train))
print("Clean test images:", N)
print("Number of identities:", K)
print("Selected tree settings:",
      selected_tree.max_depth, selected_tree.min_samples_leaf)
print("Selected boosting depth:", selected_boost.max_depth)
print("Selected boosting rounds:", len(selected_boost.estimators_))

In [ ]:
def one_hot_labels(y, classes):
    class_to_index = {cls: j for j, cls in enumerate(classes)}
    result = np.zeros((len(y), len(classes)), dtype=float)

    for i, label in enumerate(y):
        result[i, class_to_index[label]] = 1.0

    return result


def bias_variance(prediction_vectors, y_true, classes):
    """
    prediction_vectors shape: (R, N, K)
    y_true shape: (N,)
    """
    R, N, K = prediction_vectors.shape

    mean_prediction = prediction_vectors.mean(axis=0)
    target_vectors = one_hot_labels(y_true, classes)

    bias_squared = np.mean(
        np.sum((mean_prediction - target_vectors) ** 2, axis=1)
    )

    variance = np.mean(
        np.sum(
            (prediction_vectors - mean_prediction[None, :, :]) ** 2,
            axis=2
        )
    )

    return bias_squared, variance

In [ ]:
def fit_samme_for_q3(X, y, max_depth, n_estimators, random_state):
    rng = np.random.default_rng(random_state)
    classes_local = np.unique(y)
    K_local = len(classes_local)
    n = len(y)

    weights = np.ones(n, dtype=float) / n
    estimators = []
    alphas = []

    for t in range(n_estimators):
        tree = DecisionTreeClassifier(
            max_depth=max_depth,
            random_state=int(rng.integers(0, 2**31 - 1))
        )
        tree.fit(X, y, sample_weight=weights)

        pred = tree.predict(X)
        error = np.sum(weights * (pred != y))

        # Stop if learner is no better than random
        if error >= 1 - 1 / K_local:
            break

        # Perfect training classification: accept and stop
        if error <= 1e-15:
            estimators.append(tree)
            alphas.append(1.0)
            break

        alpha = (
            np.log((1 - error) / error)
            + np.log(K_local - 1)
        )

        estimators.append(tree)
        alphas.append(alpha)

        weights *= np.exp(alpha * (pred != y))
        weights /= weights.sum()

    return estimators, np.asarray(alphas)


def samme_vote_vectors(estimators, alphas, X, classes):
    scores = np.zeros((len(X), len(classes)), dtype=float)
    class_to_index = {cls: j for j, cls in enumerate(classes)}

    for tree, alpha in zip(estimators, alphas):
        pred = tree.predict(X)

        for cls, j in class_to_index.items():
            scores[:, j] += alpha * (pred == cls)

    total_alpha = np.sum(alphas)

    if total_alpha <= 0:
        return np.ones_like(scores) / len(classes)

    return scores / total_alpha

In [ ]:
R = 30
N_TREES_BAG = 100
N_ROUNDS_BOOST = len(selected_boost.estimators_)

# Get the original training indices by identity
identity_positions = {
    identity: np.where(y_train == identity)[0]
    for identity in classes
}

for identity, positions in identity_positions.items():
    assert len(positions) == 6, (
        f"Expected six original training images for identity {identity}, "
        f"found {len(positions)}."
    )

bootstrap_sets = []
rng_q3 = np.random.default_rng(3030)

for r in range(R):
    sampled_positions = []

    for identity in classes:
        original_positions = identity_positions[identity]

        sampled_positions.extend(
            rng_q3.choice(
                original_positions,
                size=6,
                replace=True
            )
        )

    bootstrap_sets.append(np.asarray(sampled_positions, dtype=int))

print("Number of bootstrap repetitions:", len(bootstrap_sets))
print("Training size per repetition:", len(bootstrap_sets[0]))

In [ ]:
R = 30
N_TREES_BAG = 100
N_ROUNDS_BOOST = len(selected_boost.estimators_)

# Reset all result lists before rerunning
tree_vectors = []
bag_vectors = []
boost_vectors = []

tree_accuracies = []
bag_accuracies = []
boost_accuracies = []

tree_times = []
bag_times = []
boost_times = []

for r, positions in enumerate(bootstrap_sets):
    X_boot = X_train[positions]
    y_boot = y_train[positions]

    # 1. Single tree: use selected settings without retuning
    start = time.perf_counter()

    tree_r = DecisionTreeClassifier(
        max_depth=selected_tree.max_depth,
        min_samples_leaf=selected_tree.min_samples_leaf,
        random_state=4000 + r
    )
    tree_r.fit(X_boot, y_boot)

    tree_pred = tree_r.predict(X_test)

    tree_vectors.append(one_hot_labels(tree_pred, classes))
    tree_accuracies.append(accuracy_score(y_test, tree_pred))
    tree_times.append(time.perf_counter() - start)

    # 2. Bagging: 100 trees trained on bootstrap samples
    start = time.perf_counter()

    bag_rng = np.random.default_rng(5000 + r)
    bag_trees_r = []

    for b in range(N_TREES_BAG):
        sample_idx = bag_rng.choice(
            len(y_boot),
            size=len(y_boot),
            replace=True
        )

        bag_tree = DecisionTreeClassifier(
            max_depth=selected_tree.max_depth,
            min_samples_leaf=selected_tree.min_samples_leaf,
            random_state=6000 + r * N_TREES_BAG + b
        )

        bag_tree.fit(X_boot[sample_idx], y_boot[sample_idx])
        bag_trees_r.append(bag_tree)

    individual_bag_preds = np.array([
        t.predict(X_test) for t in bag_trees_r
    ])  # Shape: (100, 80)

    # Correct: each column represents one identity's vote fraction
    bag_vote_vectors = np.column_stack([
        np.mean(individual_bag_preds == cls, axis=0)
        for cls in classes
    ])  # Shape: (80, 40)

    bag_pred = classes[np.argmax(bag_vote_vectors, axis=1)]

    bag_vectors.append(bag_vote_vectors)
    bag_accuracies.append(accuracy_score(y_test, bag_pred))
    bag_times.append(time.perf_counter() - start)

    # 3. SAMME boosting: selected depth and number of rounds
    start = time.perf_counter()

    estimators_r, alphas_r = fit_samme_for_q3(
        X_boot,
        y_boot,
        max_depth=selected_boost.max_depth,
        n_estimators=N_ROUNDS_BOOST,
        random_state=7000 + r
    )

    boost_vote_vectors = samme_vote_vectors(
        estimators_r,
        alphas_r,
        X_test,
        classes
    )  # Shape: (80, 40)

    boost_pred = classes[np.argmax(boost_vote_vectors, axis=1)]

    boost_vectors.append(boost_vote_vectors)
    boost_accuracies.append(accuracy_score(y_test, boost_pred))
    boost_times.append(time.perf_counter() - start)

    if (r + 1) % 5 == 0:
        print(f"Completed repetition {r + 1}/{R}")

# Convert lists into arrays
tree_vectors = np.asarray(tree_vectors)
bag_vectors = np.asarray(bag_vectors)
boost_vectors = np.asarray(boost_vectors)

print("\nPrediction vector shapes:")
print("Single tree:", tree_vectors.shape)
print("Bagging:", bag_vectors.shape)
print("Boosting:", boost_vectors.shape)

# Confirm expected dimensions
assert tree_vectors.shape == (R, len(y_test), len(classes))
assert bag_vectors.shape == (R, len(y_test), len(classes))
assert boost_vectors.shape == (R, len(y_test), len(classes))

print("\nAll prediction vector shapes are correct.")

In [ ]:
models_q3 = {
    "Selected single tree": (tree_vectors, tree_accuracies, tree_times),
    "Selected bagging (100 trees)": (bag_vectors, bag_accuracies, bag_times),
    "Selected SAMME boosting": (boost_vectors, boost_accuracies, boost_times)
}

q3a_rows = []

for model_name, (vectors, accuracies, times) in models_q3.items():
    bias_sq, variance = bias_variance(vectors, y_test, classes)

    q3a_rows.append({
        "Model": model_name,
        "Bias squared": bias_sq,
        "Variance": variance,
        "Mean test accuracy": np.mean(accuracies),
        "SD test accuracy": np.std(accuracies, ddof=1),
        "Mean fit time (seconds)": np.mean(times)
    })

q3a_results = pd.DataFrame(q3a_rows)
display(q3a_results.round(5))

3.B how depth affects bias and variance

In [ ]:
import numpy as np
import pandas as pd
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

R = len(bootstrap_sets)
N_TREES_BAG = 100

# The selected tree is unrestricted (max_depth=None).
# Use its observed fitted depth as a numeric comparison point.
selected_observed_depth = selected_tree.get_depth()

shallow_depth = max(1, selected_observed_depth // 2)
middle_depth = selected_observed_depth
deep_depth = max(middle_depth + 2, 2 * middle_depth)

depth_settings = [
    ("Shallow", shallow_depth),
    ("Selected-depth constraint", middle_depth),
    ("Deeper", deep_depth)
]

print("Original selected tree's observed depth:", selected_observed_depth)
print("Depth settings:", depth_settings)

depth_results = []

for depth_label, depth in depth_settings:
    single_vectors = []
    bag_vectors_at_depth = []

    single_accuracies = []
    bag_accuracies_at_depth = []

    leaf_counts = []
    leaf_occupancies = []
    disagreements = []

    for r, positions in enumerate(bootstrap_sets):
        X_boot = X_train[positions]
        y_boot = y_train[positions]

        # Fit one decision tree
        single_tree = DecisionTreeClassifier(
            max_depth=depth,
            min_samples_leaf=selected_leaf,
            random_state=11000 + r
        )
        single_tree.fit(X_boot, y_boot)

        single_pred = single_tree.predict(X_test)

        single_vectors.append(
            one_hot_labels(single_pred, classes)
        )
        single_accuracies.append(
            accuracy_score(y_test, single_pred)
        )

        # Measure the fitted tree's complexity on its training sample
        leaf_ids = single_tree.apply(X_boot)
        _, leaf_sizes = np.unique(leaf_ids, return_counts=True)

        leaf_counts.append(single_tree.get_n_leaves())
        leaf_occupancies.append(np.mean(leaf_sizes))

        # Fit 100 bagged trees at the same depth
        rng = np.random.default_rng(12000 + r)
        bag_trees = []

        for b in range(N_TREES_BAG):
            sample_idx = rng.choice(
                len(y_boot),
                size=len(y_boot),
                replace=True
            )

            bag_tree = DecisionTreeClassifier(
                max_depth=depth,
                min_samples_leaf=selected_leaf,
                random_state=13000 + r * N_TREES_BAG + b
            )

            bag_tree.fit(
                X_boot[sample_idx],
                y_boot[sample_idx]
            )
            bag_trees.append(bag_tree)

        # Each row = one tree; each column = one test image
        individual_preds = np.array([
            tree.predict(X_test) for tree in bag_trees
        ])

        # Soft vote fractions across the 100 hard predictions
        vote_vectors = np.column_stack([
            np.mean(individual_preds == cls, axis=0)
            for cls in classes
        ])

        bag_pred = classes[np.argmax(vote_vectors, axis=1)]

        bag_vectors_at_depth.append(vote_vectors)
        bag_accuracies_at_depth.append(
            accuracy_score(y_test, bag_pred)
        )

        # Mean pairwise disagreement among bagged trees
        pair_disagreement = []

        for a in range(N_TREES_BAG):
            for b in range(a + 1, N_TREES_BAG):
                pair_disagreement.append(
                    np.mean(individual_preds[a] != individual_preds[b])
                )

        disagreements.append(np.mean(pair_disagreement))

    # Convert to arrays for bias-variance calculations
    single_vectors = np.asarray(single_vectors)
    bag_vectors_at_depth = np.asarray(bag_vectors_at_depth)

    single_bias, single_var = bias_variance(
        single_vectors, y_test, classes
    )
    bag_bias, bag_var = bias_variance(
        bag_vectors_at_depth, y_test, classes
    )

    variance_removed = single_var - bag_var

    depth_results.append({
        "Depth setting": depth_label,
        "max_depth": depth,
        "Single bias squared": single_bias,
        "Single variance": single_var,
        "Bagging bias squared": bag_bias,
        "Bagging variance": bag_var,
        "Single mean test accuracy": np.mean(single_accuracies),
        "Bagging mean test accuracy": np.mean(bag_accuracies_at_depth),
        "Accuracy improvement": (
            np.mean(bag_accuracies_at_depth)
            - np.mean(single_accuracies)
        ),
        "Variance removed": variance_removed,
        "Relative variance reduction": (
            variance_removed / single_var
            if single_var > 0 else np.nan
        ),
        "Mean number of leaves": np.mean(leaf_counts),
        "Mean samples per leaf": np.mean(leaf_occupancies),
        "Mean pairwise disagreement": np.mean(disagreements)
    })

    print(f"Finished {depth_label}: max_depth={depth}")

q3b_results = pd.DataFrame(depth_results)

display(q3b_results.round(5))

In [ ]:
print("Depth with the largest accuracy improvement from bagging:")
display(
    q3b_results.sort_values(
        "Accuracy improvement", ascending=False
    )[
        [
            "Depth setting",
            "max_depth",
            "Single mean test accuracy",
            "Bagging mean test accuracy",
            "Accuracy improvement"
        ]
    ].head(3).round(5)
)

print("Depth with the largest absolute variance reduction:")
display(
    q3b_results.sort_values(
        "Variance removed", ascending=False
    )[
        [
            "Depth setting",
            "max_depth",
            "Single variance",
            "Bagging variance",
            "Variance removed",
            "Relative variance reduction"
        ]
    ].head(3).round(5)
)

print("Full results, including complexity and disagreement:")
display(
    q3b_results[
        [
            "Depth setting",
            "max_depth",
            "Single bias squared",
            "Single variance",
            "Bagging bias squared",
            "Bagging variance",
            "Mean number of leaves",
            "Mean samples per leaf",
            "Mean pairwise disagreement"
        ]
    ].round(5)
)

3.c result

In [ ]:
# Q3(c): Summarize results from Q3(a) and Q3(b)

print("1. Which depth benefited most from bagging?")
display(
    q3b_results.sort_values(
        "Accuracy improvement", ascending=False
    )[
        ["Depth setting", "max_depth",
         "Accuracy improvement"]
    ].round(5)
)

print("\n2. Which depth had the largest variance reduction?")
display(
    q3b_results.sort_values(
        "Variance removed", ascending=False
    )[
        ["Depth setting", "max_depth",
         "Single variance", "Bagging variance",
         "Variance removed"]
    ].round(5)
)

print("\n3. Tree complexity and disagreement:")
display(
    q3b_results[
        ["Depth setting", "max_depth",
         "Mean number of leaves", "Mean samples per leaf",
         "Mean pairwise disagreement"]
    ].round(5)
)

3.D masked test

In [ ]:
required = [
    "bootstrap_sets",
    "tree_vectors",
    "bag_vectors",
    "boost_vectors",
    "X_train", "y_train",
    "X_test", "y_test",
    "X_test_occluded", "y_test_occluded",
    "classes",
]

for name in required:
    print(f"{name}: {'available' if name in globals() else 'MISSING'}")

In [ ]:

# ============================================================
# Q3(d): CLEAN VS MASKED TEST PERFORMANCE
# Reuse all existing Q3(a) results; do not rerun Q3(a)-Q3(c).
# ============================================================

import numpy as np
import pandas as pd
import time
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

R = len(bootstrap_sets)
N_TREES_BAG = 100
N_ROUNDS_BOOST = len(selected_boost.estimators_)

masked_tree_vectors = []
masked_bag_vectors = []
masked_boost_vectors = []

masked_tree_accuracies = []
masked_bag_accuracies = []
masked_boost_accuracies = []

start = time.perf_counter()

for r, positions in enumerate(bootstrap_sets):
    X_boot = X_train[positions]
    y_boot = y_train[positions]

    # --------------------------------------------------------
    # 1. Selected single tree
    # Same settings and seed as Q3(a)
    # --------------------------------------------------------
    tree_r = DecisionTreeClassifier(
        max_depth=selected_tree.max_depth,
        min_samples_leaf=selected_tree.min_samples_leaf,
        random_state=4000 + r
    )
    tree_r.fit(X_boot, y_boot)

    pred_tree = tree_r.predict(X_test_occluded)
    masked_tree_vectors.append(
        one_hot_labels(pred_tree, classes)
    )
    masked_tree_accuracies.append(
        accuracy_score(y_test_occluded, pred_tree)
    )

    # --------------------------------------------------------
    # 2. Selected bagging model: 100 trees
    # Same bootstrap seeds and settings as Q3(a)
    # --------------------------------------------------------
    bag_rng = np.random.default_rng(5000 + r)
    bag_trees = []

    for b in range(N_TREES_BAG):
        sample_idx = bag_rng.choice(
            len(y_boot),
            size=len(y_boot),
            replace=True
        )

        bag_tree = DecisionTreeClassifier(
            max_depth=selected_tree.max_depth,
            min_samples_leaf=selected_tree.min_samples_leaf,
            random_state=6000 + r * N_TREES_BAG + b
        )
        bag_tree.fit(X_boot[sample_idx], y_boot[sample_idx])
        bag_trees.append(bag_tree)

    individual_preds = np.asarray([
        model.predict(X_test_occluded)
        for model in bag_trees
    ])

    votes_bag = np.column_stack([
        np.mean(individual_preds == cls, axis=0)
        for cls in classes
    ])

    pred_bag = classes[np.argmax(votes_bag, axis=1)]

    masked_bag_vectors.append(votes_bag)
    masked_bag_accuracies.append(
        accuracy_score(y_test_occluded, pred_bag)
    )

    # --------------------------------------------------------
    # 3. Selected SAMME boosting model
    # Same settings and seed as Q3(a)
    # --------------------------------------------------------
    estimators_r, alphas_r = fit_samme_for_q3(
        X_boot,
        y_boot,
        max_depth=selected_boost.max_depth,
        n_estimators=N_ROUNDS_BOOST,
        random_state=7000 + r
    )

    votes_boost = samme_vote_vectors(
        estimators_r,
        alphas_r,
        X_test_occluded,
        classes
    )

    pred_boost = classes[np.argmax(votes_boost, axis=1)]

    masked_boost_vectors.append(votes_boost)
    masked_boost_accuracies.append(
        accuracy_score(y_test_occluded, pred_boost)
    )

    if (r + 1) % 5 == 0:
        elapsed = (time.perf_counter() - start) / 60
        print(f"Completed {r + 1}/{R} repetitions "
              f"({elapsed:.1f} minutes)")

# Convert masked predictions into arrays
masked_tree_vectors = np.asarray(masked_tree_vectors)
masked_bag_vectors = np.asarray(masked_bag_vectors)
masked_boost_vectors = np.asarray(masked_boost_vectors)

print("\nMasked evaluation completed.")
print("Tree:", masked_tree_vectors.shape)
print("Bagging:", masked_bag_vectors.shape)
print("Boosting:", masked_boost_vectors.shape)
print(f"Runtime: {(time.perf_counter() - start) / 60:.2f} minutes")


In [ ]:

# Clean predictions and accuracies already calculated in Q3(a)
clean_results = {
    "Selected single tree": (tree_vectors, tree_accuracies),
    "Selected bagging (100 trees)": (bag_vectors, bag_accuracies),
    "Selected SAMME boosting": (boost_vectors, boost_accuracies)
}

masked_results = {
    "Selected single tree": (masked_tree_vectors, masked_tree_accuracies),
    "Selected bagging (100 trees)": (masked_bag_vectors, masked_bag_accuracies),
    "Selected SAMME boosting": (masked_boost_vectors, masked_boost_accuracies)
}

rows = []

for model_name in clean_results:
    clean_vectors, clean_acc = clean_results[model_name]
    masked_vectors, masked_acc = masked_results[model_name]

    clean_bias, clean_var = bias_variance(
        clean_vectors, y_test, classes
    )
    masked_bias, masked_var = bias_variance(
        masked_vectors, y_test_occluded, classes
    )

    rows.append({
        "Model": model_name,
        "Clean test accuracy": np.mean(clean_acc),
        "Masked test accuracy": np.mean(masked_acc),
        "Clean bias squared": clean_bias,
        "Clean variance": clean_var,
        "Masked bias squared": masked_bias,
        "Masked variance": masked_var,
        "Accuracy drop": np.mean(clean_acc) - np.mean(masked_acc)
    })

q3d_final = pd.DataFrame(rows)

print("Q3(d): Final comparison")
display(q3d_final.round(5))


In [ ]:
rows = []

for model_name in clean_results:
    clean_vectors, clean_acc = clean_results[model_name]
    masked_vectors, masked_acc = masked_results[model_name]

    clean_bias, clean_var = bias_variance(
        clean_vectors, y_test, classes
    )

    masked_bias, masked_var = bias_variance(
        masked_vectors, y_test_occluded, classes
    )

    clean_mean = np.mean(clean_acc)
    masked_mean = np.mean(masked_acc)

    rows.append({
        "Model": model_name,
        "Clean test accuracy": clean_mean,
        "Clean accuracy std": np.std(clean_acc, ddof=0),
        "Masked test accuracy": masked_mean,
        "Masked accuracy std": np.std(masked_acc, ddof=0),
        "Clean bias squared": clean_bias,
        "Clean variance": clean_var,
        "Masked bias squared": masked_bias,
        "Masked variance": masked_var,
        "Accuracy drop": clean_mean - masked_mean
    })

q3d_final = pd.DataFrame(rows)

print("Q3(d): Final comparison")
display(q3d_final.round(5))

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

models = [
    "Selected single tree",
    "Selected bagging (100 trees)",
    "Selected SAMME boosting"
]

bias = q3d_final["Clean bias squared"].to_numpy()
variance = q3d_final["Clean variance"].to_numpy()

x = np.arange(len(models))
width = 0.35

fig, ax = plt.subplots(figsize=(9, 5))

ax.bar(x - width/2, bias, width, label="Bias squared")
ax.bar(x + width/2, variance, width, label="Variance")

ax.set_xticks(x)
ax.set_xticklabels(["Single tree", "Bagging", "SAMME boosting"])
ax.set_ylabel("Measured value")
ax.set_title("Q3(a): Bias–Variance Comparison")
ax.legend()
ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
depth_results = {
    "Depth setting": ["Shallow", "Selected-depth", "Deeper"],
    "max_depth": [15, 31, 62],
    "Single bias squared": [1.00664, 0.45739, 0.38911],
    "Single variance": [0.49336, 0.67428, 0.65756],
    "Bagging bias squared": [0.68883, 0.53042, 0.52309],
    "Bagging variance": [0.01970, 0.02371, 0.02393]
}

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

depths = depth_results["max_depth"]

axes[0].plot(depths, depth_results["Single bias squared"],
             marker="o", label="Single tree")
axes[0].plot(depths, depth_results["Bagging bias squared"],
             marker="s", label="Bagging")
axes[0].set_title("Bias squared vs tree depth")
axes[0].set_xlabel("Maximum depth")
axes[0].set_ylabel("Bias squared")
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(depths, depth_results["Single variance"],
             marker="o", label="Single tree")
axes[1].plot(depths, depth_results["Bagging variance"],
             marker="s", label="Bagging")
axes[1].set_title("Variance vs tree depth")
axes[1].set_xlabel("Maximum depth")
axes[1].set_ylabel("Variance")
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
single_acc = np.array([0.25000, 0.43417, 0.47667])
bag_acc = np.array([0.62625, 0.76542, 0.77125])

labels = ["Shallow (15)", "Selected (31)", "Deeper (62)"]
improvement = bag_acc - single_acc

fig, ax = plt.subplots(figsize=(9, 5))

bars = ax.bar(labels, improvement)

for bar, value in zip(bars, improvement):
    ax.text(
        bar.get_x() + bar.get_width()/2,
        bar.get_height() + 0.008,
        f"{value:.3f}",
        ha="center",
        va="bottom"
    )

ax.set_ylabel("Accuracy improvement")
ax.set_title("Q3(c): Test Accuracy Gain from Bagging")
ax.set_ylim(0, max(improvement) + 0.08)
ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
clean_acc = np.array([0.48625, 0.76875, 0.25250])
masked_acc = np.array([0.22667, 0.57833, 0.23417])

labels = ["Single tree", "Bagging", "SAMME boosting"]

x = np.arange(len(labels))
width = 0.35

fig, ax = plt.subplots(figsize=(9, 5))

ax.bar(x - width/2, clean_acc, width, label="Clean test")
ax.bar(x + width/2, masked_acc, width, label="Masked test")

ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.set_ylabel("Mean test accuracy")
ax.set_ylim(0, 1)
ax.set_title("Q3(d): Clean vs Masked Test Accuracy")
ax.legend()
ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

depths = ["Shallow (15)", "Selected (31)", "Deeper (62)"]

single_bias = [1.00664, 0.45739, 0.38911]
bag_bias = [0.68883, 0.53042, 0.52309]

single_var = [0.49336, 0.67428, 0.65756]
bag_var = [0.01970, 0.02371, 0.02393]

single_acc = [0.25000, 0.43417, 0.47667]
bag_acc = [0.62625, 0.76542, 0.77125]

variance_removed = [0.47366, 0.65057, 0.63362]

x = np.arange(len(depths))

fig, axes = plt.subplots(2, 2, figsize=(12, 8))

# Panel 1: Bias squared
axes[0, 0].plot(x, single_bias, marker="o", label="Single tree")
axes[0, 0].plot(x, bag_bias, marker="s", label="Bagging")
axes[0, 0].set_title("Bias squared vs depth")
axes[0, 0].set_ylabel("Bias squared")
axes[0, 0].legend()
axes[0, 0].grid(alpha=0.3)

# Panel 2: Variance
axes[0, 1].plot(x, single_var, marker="o", label="Single tree")
axes[0, 1].plot(x, bag_var, marker="s", label="Bagging")
axes[0, 1].set_title("Variance vs depth")
axes[0, 1].set_ylabel("Variance")
axes[0, 1].legend()
axes[0, 1].grid(alpha=0.3)

# Panel 3: Accuracy improvement
improvement = np.array(bag_acc) - np.array(single_acc)
axes[1, 0].bar(x, improvement)
axes[1, 0].set_title("Test accuracy improvement from bagging")
axes[1, 0].set_ylabel("Accuracy gain")
axes[1, 0].set_ylim(0, 0.45)
axes[1, 0].grid(axis="y", alpha=0.3)

for i, v in enumerate(improvement):
    axes[1, 0].text(i, v + 0.01, f"{v:.3f}", ha="center")

# Panel 4: Absolute variance reduction
axes[1, 1].bar(x, variance_removed)
axes[1, 1].set_title("Variance removed by bagging")
axes[1, 1].set_ylabel("Single variance − bagging variance")
axes[1, 1].grid(axis="y", alpha=0.3)

for ax in axes.flat:
    ax.set_xticks(x)
    ax.set_xticklabels(depths, rotation=15)

plt.suptitle("Q3: Bias–Variance Tradeoff and Bagging Across Tree Depths")
plt.tight_layout()
plt.savefig("Q3_bias_variance_depth.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

models = ["Single tree", "Bagging (100)", "SAMME boosting"]
x = np.arange(len(models))
width = 0.35

clean_acc = [0.48625, 0.76875, 0.25250]
masked_acc = [0.22667, 0.57833, 0.23417]

clean_bias = [0.37819, 0.52306, 0.88890]
masked_bias = [0.64931, 0.69114, 0.89519]

clean_var = [0.22667, 0.02493, 0.02216]
masked_var = [0.75514, 0.64931, 0.89519]

# Replace the variance arrays above with the correctly aligned
# clean and masked variance columns from your q3d_final DataFrame.
# See the corrected version below if you have q3d_final available.

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].bar(x - width/2, clean_acc, width, label="Clean")
axes[0].bar(x + width/2, masked_acc, width, label="Masked")
axes[0].set_title("Test accuracy")
axes[0].set_ylabel("Mean accuracy")
axes[0].set_ylim(0, 1)
axes[0].legend()
axes[0].grid(axis="y", alpha=0.3)

axes[1].bar(x - width/2, clean_bias, width, label="Clean bias squared")
axes[1].bar(x + width/2, masked_bias, width, label="Masked bias squared")
axes[1].set_title("Bias squared under occlusion")
axes[1].set_ylabel("Bias squared")
axes[1].legend()
axes[1].grid(axis="y", alpha=0.3)

for ax in axes:
    ax.set_xticks(x)
    ax.set_xticklabels(models, rotation=15)

plt.suptitle("Q3: Clean vs Masked Test Performance")
plt.tight_layout()
plt.savefig("Q3_clean_vs_masked.png", dpi=300, bbox_inches="tight")
plt.show()